# Lab 07 — Linear SVM + char-ngrams + imbalance

Track A. This notebook reproduces the Lab6 baseline, compares it against LinearSVC variants, checks the effect of char-ngrams and `class_weight`, and adds one-vs-rest PR/threshold analysis.

## 1) Install deps

In [1]:
!pip -q install -r ../requirements.txt

## 2) Data access

In [2]:
from pathlib import Path
import json
import re
import sys

import numpy as np
import pandas as pd

LAB7_ROOT = Path('..').resolve()
LAB2_ROOT = (LAB7_ROOT.parent / 'project_lab2').resolve()
LAB5_ROOT = (LAB7_ROOT.parent / 'project_lab5').resolve()

sys.path.insert(0, str(LAB7_ROOT))

from src.baseline_cls import (
    LogRegConfig,
    build_logreg_pipeline,
    evaluate_pipeline,
    load_split_ids,
    plot_confusion_matrix,
    subset_by_ids,
)
from src.svm_experiments import SVMConfig, run_linear_svc, top_features_from_svm_pipeline
from src.threshold_eval import evaluate_thresholds, one_vs_rest_scores, plot_pr_curve, precision_recall_summary

processed_path = LAB2_ROOT / 'data' / 'processed_v2' / 'processed_v2.csv'
df = pd.read_csv(processed_path)
df['text'] = df['text'].fillna('').astype(str).str.replace(r'\\s+', ' ', regex=True).str.strip()

print('processed_v2:', processed_path)
print('rows:', len(df))
print(df[['text_id', 'label']].head(3))


processed_v2: C:\Users\maia1\data\politiekh\masters\nlp\project_lab2\data\processed_v2\processed_v2.csv
rows: 1000
    text_id                        label
0      9905  Question / Request for Help
1  10001201  Question / Request for Help
2      3099              Neutral Comment


## 3) Load split

In [3]:
train_ids = load_split_ids(LAB5_ROOT / 'data' / 'sample' / 'splits_train_ids.txt')
val_ids = load_split_ids(LAB5_ROOT / 'data' / 'sample' / 'splits_val_ids.txt')
test_ids = load_split_ids(LAB5_ROOT / 'data' / 'sample' / 'splits_test_ids.txt')

df_train = subset_by_ids(df, train_ids)
df_val = subset_by_ids(df, val_ids)
df_test = subset_by_ids(df, test_ids)

print({'train': len(df_train), 'val': len(df_val), 'test': len(df_test)})
print(df_train['label'].value_counts())


{'train': 800, 'val': 100, 'test': 100}
label
Question / Request for Help      160
Gratitude / Positive Feedback    160
Complaint / Dissatisfaction      160
Neutral Comment                  160
Suggestion / Idea                160
Name: count, dtype: int64


## 4) Reproduce Lab6 baseline

In [4]:
logreg_cfg = LogRegConfig(name='logreg_word_1_2', word_ngram_range=(1, 2), max_iter=500, random_state=42)
logreg_res = evaluate_pipeline(
    logreg_cfg.name,
    build_logreg_pipeline(logreg_cfg),
    df_train,
    df_val,
    df_test,
)

print('LogReg baseline')
print('VAL  acc={:.4f}, macro-F1={:.4f}'.format(logreg_res['val_accuracy'], logreg_res['val_macro_f1']))
print('TEST acc={:.4f}, macro-F1={:.4f}'.format(logreg_res['test_accuracy'], logreg_res['test_macro_f1']))


LogReg baseline
VAL  acc=0.5700, macro-F1=0.5741
TEST acc=0.6800, macro-F1=0.6786


## 5) Linear SVM baseline

In [5]:
svm_word_cfg = SVMConfig(name='svm_word_1_2', use_word=True, use_char=False, word_ngram_range=(1, 2), class_weight=None)
svm_word_res = run_linear_svc(svm_word_cfg, df_train, df_val, df_test)

print('LinearSVC word baseline')
print('VAL  acc={:.4f}, macro-F1={:.4f}'.format(svm_word_res['val_accuracy'], svm_word_res['val_macro_f1']))
print('TEST acc={:.4f}, macro-F1={:.4f}'.format(svm_word_res['test_accuracy'], svm_word_res['test_macro_f1']))


LinearSVC word baseline
VAL  acc=0.5900, macro-F1=0.5935
TEST acc=0.6600, macro-F1=0.6588


## 6) Linear SVM + char-ngrams

In [6]:
svm_word_char_cfg = SVMConfig(
    name='svm_word_char',
    use_word=True,
    use_char=True,
    word_ngram_range=(1, 2),
    char_ngram_range=(3, 5),
    char_analyzer='char_wb',
    class_weight=None,
)
svm_word_char_res = run_linear_svc(svm_word_char_cfg, df_train, df_val, df_test)

print('LinearSVC word+char')
print('VAL  acc={:.4f}, macro-F1={:.4f}'.format(svm_word_char_res['val_accuracy'], svm_word_char_res['val_macro_f1']))
print('TEST acc={:.4f}, macro-F1={:.4f}'.format(svm_word_char_res['test_accuracy'], svm_word_char_res['test_macro_f1']))


LinearSVC word+char
VAL  acc=0.6500, macro-F1=0.6512
TEST acc=0.7400, macro-F1=0.7355


## 7) class_weight="balanced" comparison

In [7]:
svm_word_char_bal_cfg = SVMConfig(
    name='svm_word_char_balanced',
    use_word=True,
    use_char=True,
    word_ngram_range=(1, 2),
    char_ngram_range=(3, 5),
    char_analyzer='char_wb',
    class_weight='balanced',
)
svm_word_char_bal_res = run_linear_svc(svm_word_char_bal_cfg, df_train, df_val, df_test)

results = [logreg_res, svm_word_res, svm_word_char_res, svm_word_char_bal_res]
metrics_df = pd.DataFrame([
    {
        'model': r['name'],
        'val_accuracy': r['val_accuracy'],
        'val_macro_f1': r['val_macro_f1'],
        'test_accuracy': r['test_accuracy'],
        'test_macro_f1': r['test_macro_f1'],
    }
    for r in results
]).sort_values(['val_macro_f1', 'test_macro_f1'], ascending=False).reset_index(drop=True)

print(metrics_df)
best_name = metrics_df.iloc[0]['model']
best_map = {r['name']: r for r in results}
best_res = best_map[best_name]
print()
print('Best by validation macro-F1:', best_name)


                    model  val_accuracy  val_macro_f1  test_accuracy  \
0           svm_word_char          0.65      0.651220           0.74   
1  svm_word_char_balanced          0.65      0.651220           0.74   
2            svm_word_1_2          0.59      0.593454           0.66   
3         logreg_word_1_2          0.57      0.574074           0.68   

   test_macro_f1  
0       0.735550  
1       0.735550  
2       0.658833  
3       0.678608  

Best by validation macro-F1: svm_word_char


## 8) PR-curve / threshold section

In [8]:
positive_class = 'Complaint / Dissatisfaction'
y_val_bin = (best_res['y_val'].astype(str) == positive_class).astype(int).to_numpy()
y_test_bin = (best_res['y_test'].astype(str) == positive_class).astype(int).to_numpy()

val_scores = one_vs_rest_scores(best_res['pipeline'], df_val['text'].astype(str).tolist(), positive_class)
test_scores = one_vs_rest_scores(best_res['pipeline'], df_test['text'].astype(str).tolist(), positive_class)

pr_val = precision_recall_summary(y_val_bin, val_scores)
curve_df = plot_pr_curve(y_val_bin, val_scores)
candidate_thresholds = sorted(set([0.0] + curve_df['threshold'].round(6).tolist()))
threshold_df_val = evaluate_thresholds(y_val_bin, val_scores, candidate_thresholds)
threshold_df_val = threshold_df_val.sort_values(['f1', 'precision', 'recall'], ascending=False).reset_index(drop=True)
best_threshold = float(threshold_df_val.iloc[0]['threshold'])

threshold_df_test = evaluate_thresholds(y_test_bin, test_scores, [0.0, best_threshold])

print('Positive class for OvR PR analysis:', positive_class)
print('Average precision (val): {:.4f}'.format(pr_val['average_precision']))
print('Best validation threshold by F1:', best_threshold)
print()
print('Top validation thresholds:')
print(threshold_df_val.head(10).to_string(index=False))
print()
print('Default vs tuned threshold on test:')
print(threshold_df_test.to_string(index=False))


Positive class for OvR PR analysis: Complaint / Dissatisfaction
Average precision (val): 0.6801
Best validation threshold by F1: -0.095137

Top validation thresholds:
 threshold  precision  recall       f1  predicted_positive
 -0.095137   0.714286    0.50 0.588235                  14
 -0.088512   0.714286    0.50 0.588235                  14
 -0.414379   0.500000    0.70 0.583333                  28
 -0.201747   0.611111    0.55 0.578947                  18
 -0.171221   0.611111    0.55 0.578947                  18
 -0.325975   0.520000    0.65 0.577778                  25
 -0.321926   0.520000    0.65 0.577778                  25
  0.167722   1.000000    0.40 0.571429                   8
  0.169877   1.000000    0.40 0.571429                   8
 -0.108695   0.666667    0.50 0.571429                  15

Default vs tuned threshold on test:
 threshold  precision  recall       f1  predicted_positive
 -0.095137   0.777778    0.70 0.736842                  18
  0.000000   0.812500    0.65

## 9) Confusion matrix comparison

In [9]:
labels = sorted(df_train['label'].astype(str).unique().tolist())
cm_logreg = plot_confusion_matrix(logreg_res['y_test'], logreg_res['pred_test'], labels)
cm_best = plot_confusion_matrix(best_res['y_test'], best_res['pred_test'], labels)

print('Confusion matrix: LogReg baseline')
print(cm_logreg)
print()
print('Confusion matrix: best Lab7 model ->', best_res['name'])
print(cm_best)


Confusion matrix: LogReg baseline
                                     pred::Complaint / Dissatisfaction  \
gold::Complaint / Dissatisfaction                                   15   
gold::Gratitude / Positive Feedback                                  1   
gold::Neutral Comment                                                3   
gold::Question / Request for Help                                    4   
gold::Suggestion / Idea                                              3   

                                     pred::Gratitude / Positive Feedback  \
gold::Complaint / Dissatisfaction                                      2   
gold::Gratitude / Positive Feedback                                   16   
gold::Neutral Comment                                                  4   
gold::Question / Request for Help                                      0   
gold::Suggestion / Idea                                                1   

                                     pred::Neutral Comment  \
go

## 10) Top features + error analysis

In [10]:
top_feats = top_features_from_svm_pipeline(best_res['pipeline'], top_n=10)
for cls, block in top_feats.items():
    print()
    print('=' * 80)
    print('CLASS:', cls)
    print('Top + features:')
    for feat, weight in block['top_positive']:
        print('  + {:<30} {:.4f}'.format(feat, weight))

test_df = df_test.copy().reset_index(drop=True)
test_df['pred_label'] = best_res['pred_test'].astype(str)
errors = test_df[test_df['label'].astype(str) != test_df['pred_label'].astype(str)].copy().reset_index(drop=True)

def categorize_error(text: str) -> str:
    t = str(text)
    tl = t.lower()
    wc = len(t.split())
    if wc <= 4:
        return 'short text / недостатньо контексту'
    if re.search(r'[A-Za-z]', t):
        return 'translit / slang / noisy text'
    if '?' in t and any(x in tl for x in ['не', 'жах', 'поган', 'черга', 'проблем']):
        return 'overlap класів'
    if any(x in tl for x in ['дякую', 'рекомендую', 'супер', 'погано', 'жахливо']):
        return 'overlap класів'
    if wc > 35:
        return 'noisy labels'
    return 'rare vocabulary'

def explain_error(cat: str) -> str:
    if cat == 'short text / недостатньо контексту':
        return 'Короткий текст не дає достатньо сигналу для стабільного класу.'
    if cat == 'translit / slang / noisy text':
        return 'У тексті є латинка, шум або нестандартне написання.'
    if cat == 'overlap класів':
        return 'У тексті змішані наміри, межа між класами розмита.'
    if cat == 'noisy labels':
        return 'Текст багатосигнальний або gold-мітка виглядає неоднозначною.'
    return 'Ймовірно, модель не бачила достатньо схожої рідкісної лексики.'

errors['error_category'] = errors['text'].apply(categorize_error)
errors['comment'] = errors['error_category'].apply(explain_error)
sample_errors = errors.head(10).copy()
print()
print('Total errors on test:', len(errors))
print(sample_errors[['text_id', 'label', 'pred_label', 'error_category', 'comment']])

error_out = LAB7_ROOT / 'tests' / 'error_cases_lab7.jsonl'
with error_out.open('w', encoding='utf-8') as f:
    for _, row in sample_errors.iterrows():
        rec = {
            'text_id': int(row['text_id']),
            'text': str(row['text']),
            'gold_label': str(row['label']),
            'pred_label': str(row['pred_label']),
            'error_category': str(row['error_category']),
            'comment': str(row['comment']),
        }
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')

error_category_counts = errors['error_category'].value_counts().to_dict()
print()
print('Error categories:', error_category_counts)
print('Saved:', error_out)



CLASS: Complaint / Dissatisfaction
Top + features:
  + word_tfidf__не                 1.4607
  + word_tfidf__чергу              0.7324
  + word_tfidf__нічого             0.6878
  + char_tfidf__ не                0.6425
  + word_tfidf__потім              0.6342
  + word_tfidf__поруч              0.6298
  + word_tfidf__мінус              0.5904
  + word_tfidf__що                 0.5862
  + word_tfidf__беруть             0.5825
  + word_tfidf__погано             0.5378

CLASS: Gratitude / Positive Feedback
Top + features:
  + word_tfidf__дякую              1.0373
  + word_tfidf__дуже               0.7720
  + word_tfidf__гарно              0.7148
  + word_tfidf__позитивні          0.6279
  + word_tfidf__викладачі          0.6153
  + word_tfidf__відмінний          0.5899
  + word_tfidf__задоволений        0.5747
  + word_tfidf__архітектура        0.5428
  + word_tfidf__якісно             0.5403
  + word_tfidf__школа              0.5268

CLASS: Neutral Comment
Top + features:
  + word_tfidf

## 11) Generate docs/audit_summary_lab7.md

In [11]:
docs_dir = LAB7_ROOT / 'docs'
docs_dir.mkdir(parents=True, exist_ok=True)
audit_path = docs_dir / 'audit_summary_lab7.md'
card_path = docs_dir / 'dataset_card.md'
readme_path = LAB7_ROOT / 'labs' / 'lab07' / 'README.md'

winner_row = metrics_df.iloc[0]
runner_row = metrics_df.iloc[1]
top_err = list(error_category_counts.keys())[:3]
while len(top_err) < 3:
    top_err.append('n/a')

audit_lines = [
    '# Audit summary — Lab7',
    '',
    '1. Track A multi-class classification on UAReviews categories.',
    '2. Split: reused Lab5 train/val/test = 800/100/100, seed=42.',
    '3. Lab6 baseline (LogReg word(1,2)): accuracy={:.4f}, macro-F1={:.4f}.'.format(logreg_res['test_accuracy'], logreg_res['test_macro_f1']),
    '4. Best SVM variant ({}): accuracy={:.4f}, macro-F1={:.4f}.'.format(best_res['name'], best_res['test_accuracy'], best_res['test_macro_f1']),
    '5. Char-ngrams effect: svm_word_char test macro-F1={:.4f}; balanced variant test macro-F1={:.4f}.'.format(svm_word_char_res['test_macro_f1'], svm_word_char_bal_res['test_macro_f1']),
    '6. class_weight=balanced effect: val macro-F1={:.4f} vs unbalanced {:.4f}.'.format(svm_word_char_bal_res['val_macro_f1'], svm_word_char_res['val_macro_f1']),
    '7. Top error categories: {}, {}, {}.'.format(top_err[0], top_err[1], top_err[2]),
    '8. Next fixes: dedup/near-dup cleanup, char-ngram tuning, review borderline labels.',
]
audit_path.write_text('\n'.join(audit_lines) + '\n', encoding='utf-8')

card_lines = [
    '# Dataset Card — Lab7 update',
    '',
    '## New findings',
    '- Compared LogReg and LinearSVC on the same fixed split from Lab5.',
    '- Checked whether char-ngrams help with noisy text, translit, and spelling variation.',
    '- Checked class_weight=balanced and one-vs-rest threshold behavior for Complaint / Dissatisfaction.',
    '',
    '## Risks',
    '- overlap class boundaries remain the main source of confusion',
    '- noisy text / translit still hurts lexical models',
    '- precision/recall tradeoff depends on threshold choice for binary OvR use cases',
]
card_path.write_text('\n'.join(card_lines) + '\n', encoding='utf-8')

readme_lines = [
    '# LPNU NLP — Lab 07 (Linear SVM + char-ngrams + imbalance)',
    '',
    '1. Task: compare Lab6 LogReg baseline against LinearSVC variants on the same fixed split.',
    '2. Lab6 baseline reused: TF-IDF word(1,2) + Logistic Regression.',
    '3. SVM variants: word(1,2), word+char(3,5), and word+char with class_weight="balanced".',
    '4. Imbalance note: dataset is balanced overall, so class_weight effect is expected to be small.',
    '5. PR/threshold section: one-vs-rest analysis for Complaint / Dissatisfaction using validation scores.',
    '6. Error analysis shows overlap classes, rare vocabulary, and noisy/translit texts as the main issues.',
    '7. Best model: {} (val macro-F1={:.4f}, test macro-F1={:.4f}).'.format(best_res['name'], best_res['val_macro_f1'], best_res['test_macro_f1']),
]
readme_path.write_text('\n'.join(readme_lines) + '\n', encoding='utf-8')

print('Saved:', audit_path)
print('Saved:', card_path)
print('Saved:', readme_path)


Saved: C:\Users\maia1\data\politiekh\masters\nlp\project_lab7\docs\audit_summary_lab7.md
Saved: C:\Users\maia1\data\politiekh\masters\nlp\project_lab7\docs\dataset_card.md
Saved: C:\Users\maia1\data\politiekh\masters\nlp\project_lab7\labs\lab07\README.md
